# Intelligent Document Assistant using RAG
**GUVI/HCL Capstone Project**

This notebook builds a complete Retrieval-Augmented Generation (RAG) pipeline:
PDF ingestion → cleaning → chunking → embeddings → FAISS retrieval → Gemini-based answer generation.

Run cells top to bottom (**Runtime > Run all**). First run on a new session will process the PDF from scratch;
subsequent runs will load the saved chunks/index instantly if available.


## Setup: Mount Drive and install packages

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Pillow must be upgraded BEFORE sentence-transformers is imported, and the
# runtime must be restarted after this the FIRST time you ever run this notebook.
# If you hit an ImportError on sentence_transformers below, run this cell,
# then Runtime > Restart session, then Runtime > Run all again.
!pip install -U Pillow -q
!pip install pdfplumber pypdf sentence-transformers faiss-cpu google-genai -q

In [ ]:
from sentence_transformers import SentenceTransformer
embedding_model = SentenceTransformer('all-MiniLM-L6-v2')

In [ ]:
from google.colab import userdata
from google import genai

api_key = userdata.get('GEMINI_API_KEY')
client = genai.Client(api_key=api_key)

## Phase 1: Document Ingestion
Extracts raw text from every page of a PDF using `pdfplumber`.

In [ ]:
import pdfplumber

def extract_all_pages(pdf_path: str) -> str:
    all_pages_text = []
    with pdfplumber.open(pdf_path) as pdf:
        for page in pdf.pages:
            text = page.extract_text() or ""
            all_pages_text.append(text)
    return "\n".join(all_pages_text)

## Phase 2: Cleaning and Chunking
`clean_text` removes page-number artifacts and excess whitespace.
`chunk_text` splits the cleaned text into overlapping fixed-size chunks.

In [ ]:
import re

def clean_text(raw_text: str) -> str:
    text = raw_text
    text = re.sub(r"\n\s*\d+\s*\n", "\n", text)   # remove lines that are only a page number
    text = re.sub(r"\n{3,}", "\n\n", text)            # collapse excess blank lines
    text = re.sub(r"[ \t]{2,}", " ", text)             # collapse repeated spaces/tabs
    return text.strip()

In [ ]:
def chunk_text(text: str, chunk_size: int = 500, overlap: int = 150) -> list[str]:
    chunks = []
    start = 0
    text_length = len(text)
    while start < text_length:
        end = start + chunk_size
        chunks.append(text[start:end])
        start = end - overlap
    return chunks

## Phase 3 + 4: Embeddings and FAISS Index (build once, then cache)

This cell first tries to **load** a previously saved `chunks.pkl` and `faiss_index.bin` from Drive
(instant, no reprocessing). If they don't exist yet, it builds them from the PDF and saves them
for next time — so you only pay the processing cost once.

Set `PDF_PATH` and `FORCE_REBUILD` below as needed.

In [ ]:
import os
import pickle
import numpy as np
import faiss

PDF_PATH = '/content/drive/MyDrive/rag_project/data/_10-K-2025-As-Filed.pdf'  # change if your filename differs
CHUNKS_PATH = '/content/drive/MyDrive/rag_project/outputs/chunks.pkl'
INDEX_PATH = '/content/drive/MyDrive/rag_project/outputs/faiss_index.bin'
FORCE_REBUILD = False  # set True to reprocess the PDF even if a saved version exists

if not FORCE_REBUILD and os.path.exists(CHUNKS_PATH) and os.path.exists(INDEX_PATH):
    with open(CHUNKS_PATH, 'rb') as f:
        chunks = pickle.load(f)
    index = faiss.read_index(INDEX_PATH)
    print(f"Loaded {len(chunks)} chunks and {index.ntotal} vectors from Drive (no reprocessing).")
else:
    raw_text = extract_all_pages(PDF_PATH)
    cleaned_text = clean_text(raw_text)
    chunks = chunk_text(cleaned_text, chunk_size=500, overlap=150)
    chunk_embeddings = embedding_model.encode(chunks)

    index = faiss.IndexFlatL2(chunk_embeddings.shape[1])
    index.add(np.array(chunk_embeddings))

    os.makedirs(os.path.dirname(CHUNKS_PATH), exist_ok=True)
    with open(CHUNKS_PATH, 'wb') as f:
        pickle.dump(chunks, f)
    faiss.write_index(index, INDEX_PATH)

    print(f"Built and saved {len(chunks)} chunks and {index.ntotal} vectors.")

## Retrieval helper (for inspection/debugging)
Shows the raw retrieved chunks for a question, without generating an answer.
Useful for diagnosing retrieval quality.

In [ ]:
def search_chunks(question: str, top_k: int = 3):
    question_embedding = embedding_model.encode([question])
    distances, indices = index.search(np.array(question_embedding), top_k)

    print(f"Question: {question}\n")
    for rank, idx in enumerate(indices[0]):
        print(f"--- Match {rank+1} (distance: {distances[0][rank]:.4f}) ---")
        print(chunks[idx])
        print()

## Phase 5: Answer Generation
Retrieves the top matching chunks, builds a grounded prompt, and calls Gemini.
Includes retry logic for transient server errors, and a model choice
(`gemini-3.5-flash-lite`) with a much higher free-tier daily quota than full Flash models.

In [ ]:
import time

def generate_answer(question, top_k=3, max_retries=3):
    question_embedding = embedding_model.encode([question])
    distances, indices = index.search(np.array(question_embedding), top_k)
    retrieved_chunks = [chunks[idx] for idx in indices[0]]
    context = "\n\n".join(retrieved_chunks)

    prompt = ("You are a helpful assistant answering questions based only on the provided context.\n"
              "If the answer is not in the context, say \"I don\'t have enough information to answer that.\"\n\n"
              "Context:\n" + context + "\n\n"
              "Question: " + question + "\n\n"
              "Answer:")

    for attempt in range(max_retries):
        try:
            response = client.models.generate_content(
                model="gemini-3.5-flash-lite",
                contents=prompt
            )
            return response.text
        except Exception as e:
            print("Attempt " + str(attempt + 1) + " failed: " + str(e))
            if attempt < max_retries - 1:
                time.sleep(5)
            else:
                return "Sorry, the model is currently unavailable. Please try again later."

## Test the pipeline
A few sanity-check questions, including one the document should NOT be able to answer
(proves the system is grounded, not hallucinating from general knowledge).

In [ ]:
print(generate_answer("What was Apple's total net sales for fiscal year 2025?"))

In [ ]:
print(generate_answer("What are the main risk factors Apple identifies?"))

In [ ]:
print(generate_answer("What is the capital of France?"))  # should say it doesn't know